# Setup: Generate Sample Dataset

This cell creates the required folder structure (`data/raw/` and `data/processed/`) relative to the notebook, and generates the sample CSV dataset with missing values. 
This ensures the dataset is ready for cleaning functions and saves it to `data/raw/sample_data.csv`.

In [1]:
# --- packages this notebook needs (uncomment and run once, then re-comment) ---
# !pip install numpy
# !pip install pandas

In [2]:
# --- files this notebook needs (run me first - I only report, I change nothing) ---
from pathlib import Path

ROOT = Path.cwd()          # notebooks are meant to be run from their own folder
CHECKS = [
    ("src/cleaning.py", "NEEDED", "YOU write this in the homework - the import fails until you do"),
]

print(f"Looking in: {ROOT}\n")
missing = 0
for rel, kind, note in CHECKS:
    here = (ROOT / rel).exists()
    if not here and kind == "NEEDED":
        missing += 1
    print(f"  [{'OK ' if here else 'MISS'}]  {kind:<8}  {rel:<34}  {note}")

if missing:
    print(f"\n{missing} needed file(s) missing. Put them at the paths above, relative to:\n  {ROOT}")
    print("If that folder looks wrong, you are running the notebook from the wrong place.")
else:
    print("\nAll needed files present.")

Looking in: /Users/dorislee/bootcamp_peiyun_lee/homework/homework6

  [OK ]  NEEDED    src/cleaning.py                     YOU write this in the homework - the import fails until you do

All needed files present.


In [3]:
import os
import pandas as pd
import numpy as np

# Define folder paths relative to this notebook
raw_dir = 'data/raw'
processed_dir = 'data/processed'

# Create folders if they don't exist
os.makedirs(raw_dir, exist_ok=True)
os.makedirs(processed_dir, exist_ok=True)

# Define the sample data
data = {
    'age': [34, 45, 29, 50, 38, np.nan, 41],
    'income': [55000, np.nan, 42000, 58000, np.nan, np.nan, 49000],
    'score': [0.82, 0.91, np.nan, 0.76, 0.88, 0.65, 0.79],
    'zipcode': ['90210', '10001', '60614', '94103', '73301', '12345', '94105'],
    'city': ['Beverly', 'New York', 'Chicago', 'SF', 'Austin', 'Unknown', 'San Francisco'],
    'extra_data': [np.nan, 42, np.nan, np.nan, np.nan, 5, np.nan]
}

# Create DataFrame
df = pd.DataFrame(data)

# Save to CSV in raw data folder
csv_path = os.path.join(raw_dir, 'sample_data.csv')
if not os.path.exists(csv_path):
    df.to_csv(csv_path, index=False)
    print(f'Sample dataset created and saved to {csv_path}')
else:
    print(f'File already exists at {csv_path}. Skipping CSV creation to avoid overwrite.')


File already exists at data/raw/sample_data.csv. Skipping CSV creation to avoid overwrite.


# Homework Starter — Stage 6: Data Preprocessing
Name: Doris Lee
Date: 2026-08-18

Applies the reusable functions in `src/cleaning.py` to the raw sample dataset,
saves the cleaned result, and documents the assumptions.

In [4]:
import pandas as pd

from src import cleaning   # the cleaning functions live in src/cleaning.py

## Load Raw Dataset

In [5]:
df = pd.read_csv('data/raw/sample_data.csv')
df.head()

,age,income,score,zipcode,city,extra_data
0,34.0,55000.0,0.82,90210,Beverly,NaN
1,45.0,NaN,0.91,10001,New York,42.0
2,29.0,42000.0,NaN,60614,Chicago,NaN
3,50.0,58000.0,0.76,94103,SF,NaN
4,38.0,NaN,0.88,73301,Austin,NaN


## Apply Cleaning Functions

In [6]:
# Apply the cleaning functions in a chain (each returns a new DataFrame).
df_clean = cleaning.fill_missing_median(df, ['age', 'income', 'score'])
df_clean = cleaning.drop_missing(df_clean, threshold=0.5)
df_clean = cleaning.normalize_data(df_clean, ['age', 'income', 'score'])
df_clean.head()

,age,income,score,zipcode,city
0,0.238095,0.8125,0.653846,90210,Beverly
1,0.761905,0.6250,1.000000,10001,New York
2,0.000000,0.0000,0.596154,60614,Chicago
3,1.000000,1.0000,0.423077,94103,SF
4,0.428571,0.6250,0.884615,73301,Austin


In [7]:
# --- compare original vs cleaned ---
print("Shape: original", df.shape, "-> cleaned", df_clean.shape)
print("\nMissing values (original):")
print(df.isna().sum())
print("\nMissing values (cleaned):")
print(df_clean.isna().sum())
print("\nCleaned numeric summary:")
df_clean[['age', 'income', 'score']].describe().round(3)

Shape: original (7, 6) -> cleaned (7, 5)

Missing values (original):
age           1
income        3
score         1
zipcode       0
city          0
extra_data    5
dtype: int64

Missing values (cleaned):
age        0
income     0
score      0
zipcode    0
city       0
dtype: int64

Cleaned numeric summary:


,age,income,score
count,7.000,7.000,7.000
mean,0.500,0.589,0.585
std,0.328,0.314,0.326
min,0.000,0.000,0.000
25%,0.333,0.531,0.481
50%,0.500,0.625,0.596
75%,0.667,0.719,0.769
max,1.000,1.000,1.000


## Save Cleaned Dataset

In [8]:
df_clean.to_csv('data/processed/sample_data_cleaned.csv', index=False)
print('Saved cleaned dataset to data/processed/sample_data_cleaned.csv')


Saved cleaned dataset to data/processed/sample_data_cleaned.csv


## Assumptions & Reflection

- **Median imputation** (`fill_missing_median`): median is robust to outliers,
  so it was chosen over the mean for `age`, `income`, `score`.
- **Column dropping** (`drop_missing`, threshold 0.5): `extra_data` is 5/7
  (71%) missing, so it was removed — imputing that much would be unreliable.
- **Min-max scaling** (`normalize_data`): puts `age`, `income`, `score` on a
  common [0, 1] scale; preserves each distribution's shape (vs. z-score).
- **Not addressed yet**: `city` mixes `SF` and `San Francisco` for the same
  place and contains `Unknown` — a categorical-cleaning step for a later stage.
